# Medic AI: Healthcare Document RAG Assistant

Run the cells **top to bottom**.

1. Setup (install, Drive, paths, API key)
2. Load and chunk the FDA drug labels
3. Vector store (Chroma + local HuggingFace embeddings)
4. Gemini + RAG question answering
5. SQLite metadata and the 22-question evaluation
6. Agent upgrade (3 tools + agent test suite)
7. Gradio chat app

**Quota note:** the Gemini free tier has two limits per model: a few calls per minute (about 5) and a daily cap (about 20). Cells that call Gemini are switched off by a flag, or are resumable and pause between calls. A per-minute limit is retried automatically; a daily limit stops the cell cleanly so you can run it again tomorrow.

## 1. Setup

### Installation

In [1]:
# One install cell for the whole project.
!pip install langchain langchain-community langchain-text-splitters langchain-google-genai langchain-huggingface chromadb pypdf sentence-transformers langgraph gradio -q

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.62.1 which is incompatible.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
google-adk 2.7.1 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.45.2 which is incompatible.
google-adk 2.7.1 requires opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.45.2 which is incompatible.


### Mount Google Drive

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


### Project paths and document check

In [3]:
import os

base_dir = '/content/drive/MyDrive/Healthcare_AI_RAG_Assistant'
doc_path = os.path.join(base_dir, 'FDA Drug Levels PDF')
chroma_path = os.path.join(base_dir, 'chroma_db_local')
db_path = os.path.join(base_dir, 'project_metadata.db')

pdf_files = sorted(f for f in os.listdir(doc_path) if f.endswith('.pdf'))
print(f"Found {len(pdf_files)} PDF files:")
for f in pdf_files:
    print(" -", f)

Found 15 PDF files:
 - albuterol 20260626_9e6e982b-7dfe-45b8-aa76-754e8d2a8ef6.pdf
 - amoxicillin 20241004_42f935ca-45ba-4b3e-8b40-9b2d4eb56281.pdf
 - atorvastatin 20250926_73cefb0b-1747-4fe0-b252-377f4ed94c6f.pdf
 - citalopram 20251009_2f815b0c-6da7-cb61-e89c-7de628092d0d.pdf
 - gabapentin 20231218_7538e255-95df-4991-aae0-a90d709f1e95.pdf
 - hydrochlorothiazide 20200929_841b7962-bf5d-4c45-b594-80b17ba5187d.pdf
 - ibuprofen 20191227_5c65b691-b795-44a5-abb7-931cd772f1a1.pdf
 - levothyroxine 20260327_d28d7202-1278-0486-e053-2995a90af286.pdf
 - lisinopril 20240830_387068e7-7fb9-44f3-ae1c-e45ebc755a3e.pdf
 - losartan 20260702_a8817f22-8478-49b9-8354-ab27ca222c6b.pdf
 - metformin 20170208_2d98aea3-35ba-447a-b88f-a5a20b612b2f.pdf
 - metoprolol 20251108_be1c686e-37a5-4a53-945b-68a6ead35134.pdf
 - omeprazole 20260403_4466f17c-c27b-8393-e063-6394a90a803d.pdf
 - sertraline 20260214_a5c49e6e-b8e0-4a75-99e6-b88e323ce653.pdf
 - simvastatin 20240406_751b79c6-bba8-4d29-b69c-a2ac6453fec8.pdf


### Load the Gemini API key (stored as a Colab secret)

In [4]:
from google.colab import userdata
api_key = userdata.get('GEMINI_API_KEY')

## 2. Load and chunk the documents

### Sanity check on ONE document

In [5]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=100
)

sample = next((f for f in pdf_files if f.startswith('ibuprofen')), pdf_files[0])
pages = PyPDFLoader(os.path.join(doc_path, sample)).load()
print(f"Sample file: {sample}")
print(pages[0].page_content[:500])

sample_chunks = text_splitter.split_documents(pages)
print(f"\nNumber of chunks: {len(sample_chunks)}")
print("---")
print(sample_chunks[0].page_content)

/tmp/ipykernel_3498/1199579738.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


Sample file: ibuprofen 20191227_5c65b691-b795-44a5-abb7-931cd772f1a1.pdf
ASSURED IBUPROFEN- ibuprofen tablet, film coated 
 
Spirit Pharmaceutical LLC
----------
Assured™ Ibuprofen Tablets
Drug Facts
Active ingredient (in each tablet)
Ibuprofen 200 mg (NSAID)*
Purpose
Pain reliever/fever reducer
Uses
temporarily relieves minor aches and pains due to:
headache
toothache
backache
menstrual cramps
the common cold
muscular aches
minor pain of arthritis
temporarily reduces fever
Warnings
Allergy alert
Ibuprofen may cause a severe allergic reaction, especially in people al

Number of chunks: 10
---
ASSURED IBUPROFEN- ibuprofen tablet, film coated 
 
Spirit Pharmaceutical LLC
----------
Assured™ Ibuprofen Tablets
Drug Facts
Active ingredient (in each tablet)
Ibuprofen 200 mg (NSAID)*
Purpose
Pain reliever/fever reducer
Uses
temporarily relieves minor aches and pains due to:
headache
toothache
backache
menstrual cramps
the common cold
muscular aches
minor pain of arthritis
temporarily reduces 

### Load and chunk ALL PDFs

In [6]:
all_chunks = []

for filename in pdf_files:
    pages = PyPDFLoader(os.path.join(doc_path, filename)).load()
    doc_chunks = text_splitter.split_documents(pages)
    all_chunks.extend(doc_chunks)
    print(f"{filename}: {len(doc_chunks)} chunks")

print(f"\nTotal chunks from all documents: {len(all_chunks)}")

albuterol 20260626_9e6e982b-7dfe-45b8-aa76-754e8d2a8ef6.pdf: 61 chunks
amoxicillin 20241004_42f935ca-45ba-4b3e-8b40-9b2d4eb56281.pdf: 64 chunks
atorvastatin 20250926_73cefb0b-1747-4fe0-b252-377f4ed94c6f.pdf: 148 chunks
citalopram 20251009_2f815b0c-6da7-cb61-e89c-7de628092d0d.pdf: 132 chunks
gabapentin 20231218_7538e255-95df-4991-aae0-a90d709f1e95.pdf: 140 chunks
hydrochlorothiazide 20200929_841b7962-bf5d-4c45-b594-80b17ba5187d.pdf: 33 chunks
ibuprofen 20191227_5c65b691-b795-44a5-abb7-931cd772f1a1.pdf: 10 chunks
levothyroxine 20260327_d28d7202-1278-0486-e053-2995a90af286.pdf: 127 chunks
lisinopril 20240830_387068e7-7fb9-44f3-ae1c-e45ebc755a3e.pdf: 99 chunks
losartan 20260702_a8817f22-8478-49b9-8354-ab27ca222c6b.pdf: 125 chunks
metformin 20170208_2d98aea3-35ba-447a-b88f-a5a20b612b2f.pdf: 112 chunks
metoprolol 20251108_be1c686e-37a5-4a53-945b-68a6ead35134.pdf: 54 chunks
omeprazole 20260403_4466f17c-c27b-8393-e063-6394a90a803d.pdf: 9 chunks
sertraline 20260214_a5c49e6e-b8e0-4a75-99e6-b88e3

## 3. Embeddings and vector store

Local HuggingFace embeddings (no API quota). The cell checks for an existing store first, so re-running it never creates duplicate vectors.

In [7]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma   # a deprecation warning here is harmless

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

vectorstore = Chroma(
    persist_directory=chroma_path,
    embedding_function=embeddings
)

existing_count = vectorstore._collection.count()
print(f"Existing vectors: {existing_count}")

if existing_count == 0:
    vectorstore = Chroma.from_documents(
        documents=all_chunks,
        embedding=embeddings,
        persist_directory=chroma_path
    )
    print("Created new vector store")
else:
    print("Loaded existing vector store - no re-embedding needed")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

... (5 outputs left out)

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

/tmp/ipykernel_3498/399983095.py:8: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vectorstore = Chroma(


Existing vectors: 1406
Loaded existing vector store - no re-embedding needed


### Test retrieval (no LLM yet)

In [8]:
query = "What are the side effects of ibuprofen?"
results = vectorstore.similarity_search(query, k=3)

for i, doc in enumerate(results):
    print(f"--- Result {i+1} ---")
    print(f"Source: {doc.metadata.get('source', 'unknown')}")
    print(doc.page_content[:300])
    print()

--- Result 1 ---
Source: /content/drive/MyDrive/Healthcare_AI_RAG_Assistant/FDA Drug Levels PDF/ibuprofen 20191227_5c65b691-b795-44a5-abb7-931cd772f1a1.pdf
ASSURED IBUPROFEN- ibuprofen tablet, film coated 
 
Spirit Pharmaceutical LLC
----------
Assured™ Ibuprofen Tablets
Drug Facts
Active ingredient (in each tablet)
Ibuprofen 200 mg (NSAID)*
Purpose
Pain reliever/fever reducer
Uses
temporarily relieves minor aches and pains due to:
headache
toothache
b

--- Result 2 ---
Source: /content/drive/MyDrive/Healthcare_AI_RAG_Assistant/FDA Drug Levels PDF/ibuprofen 20191227_5c65b691-b795-44a5-abb7-931cd772f1a1.pdf
stomach pain or upset gets worse or lasts
redness or swelling is present in the painful area
any new symptoms appear
If pregnant or breast-feeding,
 ask a health professional before use. It is especially important not to
use ibuprofen during the last 3 months of pregnancy unless definitely directed 

--- Result 3 ---
Source: /content/drive/MyDrive/Healthcare_AI_RAG_Assistant/FDA Drug

## 4. Gemini and RAG question answering

In [9]:
from langchain_google_genai import ChatGoogleGenerativeAI

MODEL_NAME = "gemini-3.6-flash"   # if Google retires it, change the name here (see the optional cell below)

#max_retries=1 means "no automatic retries". The default (6) silently re-sends a failed call up to 5 more times,
# and every retry counts against the free-tier quota. call_with_retry (below) handles waiting instead.
llm = ChatGoogleGenerativeAI(
    model=MODEL_NAME,
    google_api_key=api_key,
    max_retries=1
)

def answer_text(response):
    """Gemini returns either a plain string or a list of parts; handle both."""
    content = response.content
    if isinstance(content, str):
        return content
    return "".join(p.get("text", "") for p in content if isinstance(p, dict))

def format_context(docs):
    return "\n\n".join(
        f"[Source: {d.metadata.get('source', 'unknown').split('/')[-1]}]\n{d.page_content}"
        for d in docs
    )

RAG_PROMPT = """Answer the question using ONLY the context below. If the context doesn't contain the answer, say so. Cite which source document(s) you used.

Context:
{context}

Question: {question}

Answer:"""

def ask_question(query, k=3):
    docs = vectorstore.similarity_search(query, k=k)
    prompt = RAG_PROMPT.format(context=format_context(docs), question=query)
    return answer_text(llm.invoke(prompt))

# ---- rate-limit helpers (the free tier has BOTH a per-minute limit and a per-day limit)
import time

def is_rate_limit(e):
    s = str(e)
    return "RESOURCE_EXHAUSTED" in s or "429" in s

def is_daily_limit(e):
    return "PerDay" in str(e)   # the quota id in Google's error says PerDay or PerMinute

def call_with_retry(fn, *args, max_tries=3, wait=60):
    """Retry on the per-minute limit (wait, then try again). A daily limit is raised immediately."""
    for attempt in range(max_tries):
        try:
            return fn(*args)
        except Exception as e:
            if is_rate_limit(e) and not is_daily_limit(e) and attempt < max_tries - 1:
                print(f"  Per-minute limit hit. Waiting {wait}s, then retrying ({attempt + 2}/{max_tries})...")
                time.sleep(wait)
            else:
                raise

def limit_message(e):
    return "daily quota reached" if is_daily_limit(e) else "per-minute limit still hit after retries"

### Optional troubleshooting: list available Gemini models

In [ ]:
# Only needed if the cell above fails with a 404 "model not found".
# Listing models does NOT use your generation quota.
try:
    from google import genai
    client = genai.Client(api_key=api_key)
    for m in client.models.list():
        actions = getattr(m, "supported_actions", None) or []
        if "generateContent" in actions:
            print(m.name)
except Exception as e:
    print("Could not list models:", e)

### Manual demo questions (switched off by default to save quota)

In [ ]:
RUN_DEMO_QUESTIONS = False   # each question uses 1 Gemini call; set True only when you have quota to spare

demo_questions = [
    "What are the side effects of ibuprofen?",
    "What is metformin used for?",
    "What is the recommended dosage of amoxicillin?",
    "Which of these medications can cause dizziness?",
    "What is the recommended dosage of penicillin?",
    "Is this medication safe?",
]

if RUN_DEMO_QUESTIONS:
    for q in demo_questions:
        print("Q:", q)
        print(ask_question(q))
        print("---")
else:
    print("Skipped (RUN_DEMO_QUESTIONS = False)")

Skipped (RUN_DEMO_QUESTIONS = False)


## 5. SQLite: document metadata and evaluation log

### Connect and create tables

In [10]:
import sqlite3
from collections import Counter
from datetime import datetime

conn = sqlite3.connect(db_path)
cursor = conn.cursor()

cursor.execute('''
CREATE TABLE IF NOT EXISTS documents (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    filename TEXT NOT NULL,
    filepath TEXT NOT NULL,
    chunk_count INTEGER,
    file_size_kb REAL,
    date_processed TEXT
)
''')

cursor.execute('''
CREATE TABLE IF NOT EXISTS evaluations (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    question TEXT NOT NULL,
    expected_behavior TEXT,
    actual_answer TEXT,
    passed TEXT,
    date_tested TEXT
)
''')

conn.commit()
print("Tables 'documents' and 'evaluations' are ready")

Tables 'documents' and 'evaluations' are ready


### Populate the documents table
Safe to re-run: it clears the table first, then re-inserts. Chunk counts come from `all_chunks`, so the PDFs are not re-read.

In [11]:
chunk_counts = Counter(c.metadata.get('source') for c in all_chunks)

cursor.execute('DELETE FROM documents')

for filename in pdf_files:
    filepath = os.path.join(doc_path, filename)
    file_size_kb = round(os.path.getsize(filepath) / 1024, 2)
    cursor.execute('''
    INSERT INTO documents (filename, filepath, chunk_count, file_size_kb, date_processed)
    VALUES (?, ?, ?, ?, ?)
    ''', (filename, filepath, chunk_counts.get(filepath, 0), file_size_kb, datetime.now().isoformat()))

conn.commit()
print(f"Metadata inserted for {len(pdf_files)} documents")

Metadata inserted for 15 documents


### Verify with a SQL query

In [12]:
cursor.execute('SELECT filename, chunk_count, file_size_kb FROM documents ORDER BY chunk_count DESC')
for row in cursor.fetchall():
    print(row)

('sertraline 20260214_a5c49e6e-b8e0-4a75-99e6-b88e323ce653.pdf', 149, 554.65)
('atorvastatin 20250926_73cefb0b-1747-4fe0-b252-377f4ed94c6f.pdf', 148, 944.25)
('simvastatin 20240406_751b79c6-bba8-4d29-b69c-a2ac6453fec8.pdf', 143, 815.67)
('gabapentin 20231218_7538e255-95df-4991-aae0-a90d709f1e95.pdf', 140, 812.0)
('citalopram 20251009_2f815b0c-6da7-cb61-e89c-7de628092d0d.pdf', 132, 332.71)
('levothyroxine 20260327_d28d7202-1278-0486-e053-2995a90af286.pdf', 127, 3314.17)
('losartan 20260702_a8817f22-8478-49b9-8354-ab27ca222c6b.pdf', 125, 4575.68)
('metformin 20170208_2d98aea3-35ba-447a-b88f-a5a20b612b2f.pdf', 112, 432.09)
('lisinopril 20240830_387068e7-7fb9-44f3-ae1c-e45ebc755a3e.pdf', 99, 575.9)
('amoxicillin 20241004_42f935ca-45ba-4b3e-8b40-9b2d4eb56281.pdf', 64, 397.74)
('albuterol 20260626_9e6e982b-7dfe-45b8-aa76-754e8d2a8ef6.pdf', 61, 363.53)
('metoprolol 20251108_be1c686e-37a5-4a53-945b-68a6ead35134.pdf', 54, 328.22)
('hydrochlorothiazide 20200929_841b7962-bf5d-4c45-b594-80b17ba518

### Evaluation: all 22 test questions

In [13]:
test_questions = [
    # ---- first batch (ids 1-12)
    {"question": "What is albuterol used for?",
     "expected_behavior": "Should describe bronchodilator use for asthma/breathing issues, cite albuterol.pdf"},
    {"question": "What are the warnings for lisinopril during pregnancy?",
     "expected_behavior": "Should mention fetal toxicity boxed warning, cite lisinopril.pdf"},
    {"question": "What is the dosage of levothyroxine for adults?",
     "expected_behavior": "Should give a specific dosage range, cite levothyroxine.pdf"},
    {"question": "Can hydrochlorothiazide cause electrolyte imbalances?",
     "expected_behavior": "Should mention electrolyte-related warnings, cite hydrochlorothiazide.pdf"},
    {"question": "What is gabapentin prescribed for?",
     "expected_behavior": "Should describe seizure/nerve pain use, cite gabapentin.pdf"},
    {"question": "What is the recommended dosage of aspirin?",
     "expected_behavior": "Should say context doesn't cover aspirin (not in document set) - tests hallucination resistance"},
    {"question": "Are there any drug interactions with simvastatin?",
     "expected_behavior": "Should list specific interaction warnings, cite simvastatin.pdf"},
    {"question": "What is losartan used to treat?",
     "expected_behavior": "Should describe blood pressure/hypertension use, cite losartan.pdf"},
    {"question": "What are the side effects of sertraline?",
     "expected_behavior": "Should list specific side effects, cite sertraline.pdf"},
    {"question": "Compare the uses of metoprolol and losartan",
     "expected_behavior": "Should distinguish both drugs correctly without merging incorrect info - tests cross-document handling"},
    {"question": "What is omeprazole used for?",
     "expected_behavior": "Should describe acid reflux/PPI use, cite omeprazole.pdf"},
    {"question": "What is the storage information for citalopram?",
     "expected_behavior": "Should give storage instructions if present, or admit if not covered"},

    # ---- second batch (ids 13-22)
    {"question": "What is metformin used for?",
     "expected_behavior": "Should describe type 2 diabetes/glucose control use, cite metformin.pdf"},
    {"question": "What is the maximum daily dose of ibuprofen?",
     "expected_behavior": "Should give specific dosage limit, cite ibuprofen.pdf"},
    {"question": "What conditions is gabapentin approved to treat?",
     "expected_behavior": "Should describe seizure/nerve pain use more completely than the earlier partial answer - retest of a known gap"},
    {"question": "What is the treatment for a Tylenol overdose?",
     "expected_behavior": "Should say context doesn't cover Tylenol/acetaminophen (not in document set) - hallucination resistance test"},
    {"question": "Is metformin safe to take during pregnancy?",
     "expected_behavior": "Should answer based only on document content, not general medical knowledge - tests grounding"},
    {"question": "What is losartan's maximum recommended dose?",
     "expected_behavior": "Should give specific dosage if present, cite losartan.pdf"},
    {"question": "What are the storage requirements for metoprolol?",
     "expected_behavior": "Should give storage instructions if present, cite metoprolol.pdf"},
    {"question": "How does albuterol compare to metoprolol in how they're administered?",
     "expected_behavior": "Should distinguish inhalation vs. injection/oral routes without merging incorrect details - cross-document test"},
    {"question": "What should patients avoid while taking simvastatin?",
     "expected_behavior": "Should mention specific dietary/drug interaction warnings, cite simvastatin.pdf"},
    {"question": "Does this medication cause weight gain?",
     "expected_behavior": "Ambiguous - no drug named - should ask for clarification or note it can't determine which drug is meant"},
]

print(f"Defined {len(test_questions)} test questions")

Defined 22 test questions


### Run the evaluation (resumable and duplicate-safe)
Questions already logged in SQLite are skipped by their text, so re-running this cell never creates duplicates and never spends quota on finished questions.

In [ ]:
MAX_PER_RUN = 10      # keeps one run safely under the free-tier daily quota
PAUSE_SECONDS = 13    # the free tier allows about 5 calls per minute, so pause between questions

cursor.execute("SELECT question FROM evaluations")
already_logged = {row[0] for row in cursor.fetchall()}
pending = [t for t in test_questions if t["question"] not in already_logged]
print(f"Already logged: {len(already_logged)} | pending: {len(pending)}")

for test in pending[:MAX_PER_RUN]:
    try:
        answer = call_with_retry(ask_question, test["question"])
    except Exception as e:
        if is_rate_limit(e):
            print(f"Stopping: {limit_message(e)}. Run this cell again later.")
            break
        raise

    print(f"Q: {test['question']}")
    print(f"A: {answer[:300]}")
    print("---")

    cursor.execute('''
    INSERT INTO evaluations (question, expected_behavior, actual_answer, passed, date_tested)
    VALUES (?, ?, ?, ?, ?)
    ''', (test["question"], test["expected_behavior"], answer, "TBD", datetime.now().isoformat()))
    conn.commit()
    time.sleep(PAUSE_SECONDS)

cursor.execute("SELECT COUNT(*) FROM evaluations")
print(f"\nTotal logged so far: {cursor.fetchone()[0]}")

### Review the answers

In [14]:
cursor.execute("SELECT id, question, expected_behavior, actual_answer FROM evaluations ORDER BY id")
for row in cursor.fetchall():
    print(f"ID: {row[0]}")
    print(f"Q: {row[1]}")
    print(f"Expected: {row[2]}")
    print(f"Answer: {row[3][:200]}")
    print("---")

ID: 1
Q: What is albuterol used for?
Expected: Should describe bronchodilator use for asthma/breathing issues, cite albuterol.pdf
Answer: Based on the provided context, Albuterol Inhalation Solution is used for:

* The relief of bronchospasms caused by asthma in children ages 2 to 12 years. 
* Relief of bronchospasm during labor, restri
---
ID: 2
Q: What are the warnings for lisinopril during pregnancy?
Expected: Should mention fetal toxicity boxed warning, cite lisinopril.pdf
Answer: Based on the provided context, the warnings for lisinopril regarding pregnancy include:

* **Fetal Toxicity / Risk of Harm:** Lisinopril poses a potential risk to a fetus and can cause fetal harm when
---
ID: 3
Q: What is the dosage of levothyroxine for adults?
Expected: Should give a specific dosage range, cite levothyroxine.pdf
Answer: Based on the provided context, the dosage of Levothyroxine Sodium capsules for adults with primary hypothyroidism is as follows:

* **Otherwise healthy, non-elderly adult

### Mark pass / partial / fail
These are the verdicts assigned after reading each answer. Re-running is safe (it just sets the same values). For any new question, read its answer first, then add its id here.

In [15]:
all_updates = {
    1: 'PASS', 2: 'PASS', 3: 'PASS', 4: 'PASS',
    5: 'PARTIAL', 6: 'PASS', 7: 'PASS', 8: 'PASS',
    9: 'PASS', 10: 'PARTIAL', 11: 'PASS', 12: 'PASS',
    13: 'PASS', 14: 'PARTIAL', 15: 'PARTIAL', 16: 'PASS', 17: 'PASS',
    18: 'PASS', 19: 'PASS', 20: 'PASS', 21: 'PASS', 22: 'PASS'
}

for id_, verdict in all_updates.items():
    cursor.execute("UPDATE evaluations SET passed = ? WHERE id = ?", (verdict, id_))

conn.commit()
print(f"{len(all_updates)} evaluation verdicts saved")

22 evaluation verdicts saved


### Evaluation summary (good for a README screenshot)

In [16]:
cursor.execute("SELECT id, question, passed FROM evaluations ORDER BY id")
for row in cursor.fetchall():
    print(f"ID {row[0]}: [{row[2]}] {row[1]}")

cursor.execute("SELECT COUNT(*) FROM documents")
doc_count = cursor.fetchone()[0]
cursor.execute("SELECT COUNT(*) FROM evaluations")
eval_count = cursor.fetchone()[0]

print(f"\nDocuments processed: {doc_count}")
print(f"Total evaluation questions: {eval_count}")
print("Results breakdown:")
cursor.execute("SELECT passed, COUNT(*) FROM evaluations GROUP BY passed")
for status, count in cursor.fetchall():
    print(f"  {status}: {count}")

ID 1: [PASS] What is albuterol used for?
ID 2: [PASS] What are the warnings for lisinopril during pregnancy?
ID 3: [PASS] What is the dosage of levothyroxine for adults?
ID 4: [PASS] Can hydrochlorothiazide cause electrolyte imbalances?
ID 5: [PARTIAL] What is gabapentin prescribed for?
ID 6: [PASS] What is the recommended dosage of aspirin?
ID 7: [PASS] Are there any drug interactions with simvastatin?
ID 8: [PASS] What is losartan used to treat?
ID 9: [PASS] What are the side effects of sertraline?
ID 10: [PARTIAL] Compare the uses of metoprolol and losartan
ID 11: [PASS] What is omeprazole used for?
ID 12: [PASS] What is the storage information for citalopram?
ID 13: [PASS] What is metformin used for?
ID 14: [PARTIAL] What is the maximum daily dose of ibuprofen?
ID 15: [PARTIAL] What conditions is gabapentin approved to treat?
ID 16: [PASS] What is the treatment for a Tylenol overdose?
ID 17: [PASS] Is metformin safe to take during pregnancy?
ID 18: [PASS] What is losartan's maximum

## 6. Agent upgrade: tools + tool-calling agent

The plain RAG pipeline always does the same thing (retrieve, then answer). The agent instead **decides** which tool to use:

| Tool | What it does |
|---|---|
| `search_drug_labels` | the existing retriever, wrapped as a tool |
| `compare_drugs` | pulls the same topic from two drugs' own labels so they never get mixed up |
| `dose_schedule` | time arithmetic only (never recommends a dose) |

The same guardrails apply: answers must come from tool results, cite sources, and refuse out-of-scope questions.

### Define the tools (no Gemini calls)

In [17]:
from datetime import timedelta
from langchain_core.tools import tool

# drug name -> full PDF path (filenames start with the drug name)
drug_sources = {f.split()[0].lower(): os.path.join(doc_path, f) for f in pdf_files}

def format_docs(docs):
    return format_context(docs) if docs else "No matching text found in this label."

@tool
def search_drug_labels(query: str) -> str:
    """Search all FDA drug labels for information (uses, dosage, warnings,
    side effects, interactions, storage). Use this for any question about a
    single drug or a general drug-label question."""
    return format_docs(vectorstore.similarity_search(query, k=3))

@tool
def compare_drugs(drug_a: str, drug_b: str, topic: str) -> str:
    """Compare two specific drugs on one topic (for example 'warnings',
    'side effects', 'drug interactions'). Retrieves text from each drug's own
    label so the two drugs are not mixed up. Use plain drug names like 'lisinopril'."""
    sections = []
    for drug in (drug_a, drug_b):
        path = drug_sources.get(drug.strip().lower())
        if path is None:
            sections.append(f"No label available for '{drug}'. Available drugs: {', '.join(sorted(drug_sources))}")
            continue
        docs = vectorstore.similarity_search(f"{drug} {topic}", k=2, filter={"source": path})
        sections.append(f"=== {drug.strip().upper()} ===\n{format_docs(docs)}")
    return "\n\n".join(sections)

@tool
def dose_schedule(first_dose_time: str, interval_hours: float, number_of_doses: int) -> str:
    """Time arithmetic only. Given a first dose time in 24-hour HH:MM format,
    an interval in hours and a number of doses (max 10), list the dose times.
    Does NOT decide or recommend any dose or interval."""
    try:
        start = datetime.strptime(first_dose_time.strip(), "%H:%M")
    except ValueError:
        return "Invalid time. Use 24-hour HH:MM, for example 08:00."
    if interval_hours <= 0:
        return "The interval must be greater than 0 hours."
    n = max(1, min(int(number_of_doses), 10))
    times = [(start + timedelta(hours=interval_hours * i)).strftime("%H:%M") for i in range(n)]
    return ", ".join(times)

tools = [search_drug_labels, compare_drugs, dose_schedule]
print("Tools ready:", [t.name for t in tools])

Tools ready: ['search_drug_labels', 'compare_drugs', 'dose_schedule']


### Test the tools directly (no Gemini calls, no quota)

In [18]:
print("Drugs available:", sorted(drug_sources))

print("\ndose_schedule:", dose_schedule.invoke(
    {"first_dose_time": "07:00", "interval_hours": 8, "number_of_doses": 3}))

# make sure the per-drug filter really finds text for both drugs
for d in ("lisinopril", "losartan"):
    n = len(vectorstore.similarity_search(f"{d} warnings", k=2, filter={"source": drug_sources[d]}))
    print(f"filter check - {d}: {n} chunks", "OK" if n else "PROBLEM: 0 chunks (check source paths)")

print("\nsearch_drug_labels preview:")
print(search_drug_labels.invoke({"query": "sertraline contraindications"})[:400])

print("\ncompare_drugs preview:")
print(compare_drugs.invoke({"drug_a": "lisinopril", "drug_b": "losartan", "topic": "warnings"})[:600])

Drugs available: ['albuterol', 'amoxicillin', 'atorvastatin', 'citalopram', 'gabapentin', 'hydrochlorothiazide', 'ibuprofen', 'levothyroxine', 'lisinopril', 'losartan', 'metformin', 'metoprolol', 'omeprazole', 'sertraline', 'simvastatin']

dose_schedule: 07:00, 15:00, 23:00
filter check - lisinopril: 2 chunks OK
filter check - losartan: 2 chunks OK

search_drug_labels preview:
[Source: sertraline 20260214_a5c49e6e-b8e0-4a75-99e6-b88e323ce653.pdf]
purpura, arthralgia, decreased weight, muscle twitching, and anxiety.
Other Adverse Reactions Observed During the Premarketing Evaluation of Sertraline
Other infrequent adverse reactions, not described elsewhere in the prescribing
information, occurring at an incidence of < 2% in patients treated with sertraline were:
Cardiac d

compare_drugs preview:
=== LISINOPRIL ===
[Source: lisinopril 20240830_387068e7-7fb9-44f3-ae1c-e45ebc755a3e.pdf]
decreases of approximately 0.4 g% and 1.3 vol%, respectively) occurred frequently in
patients treated wit

### Build the agent

In [19]:
try:
    from langchain.agents import create_agent
except ImportError as e:
    raise ImportError(
        "create_agent not found. Run: !pip install -U langchain langgraph  then Runtime > Restart session, and re-run from the top."
    ) from e

system_prompt = f"""You are Medic AI, an assistant for FDA drug label questions.
Drug labels available: {', '.join(sorted(drug_sources))}.

Rules:
- Answer ONLY from tool results. Cite source filenames.
- If the labels do not contain the answer, or the drug is not in the list above, say so. Do not use outside knowledge.
- Never give personal medical advice or recommend a dose for a specific person. Point to the label and a healthcare professional.
- Use dose_schedule only for time arithmetic, and only with an interval that the user or the label states."""

agent = create_agent(model=llm, tools=tools, system_prompt=system_prompt)

def run_agent(question):
    """Run the agent. Returns (final answer text, list of tool names it called)."""
    result = agent.invoke({"messages": [("user", question)]})
    messages = result["messages"]
    tools_used = []
    for m in messages:
        for call in (getattr(m, "tool_calls", None) or []):
            tools_used.append(call["name"])
    return answer_text(messages[-1]), tools_used

### Switching the agent to Groq

The agent built in the previous cell runs on Gemini. Gemini's free-tier quota ran out during this project, so the agent evaluation was run on Groq instead. The next cells install `langchain-groq`, check which tool-use models the API key can use, and rebuild the agent on `openai/gpt-oss-120b` with low reasoning effort, to stay under the free tier's limit of 8,000 tokens per minute reported by the API. The tools, system prompt and guardrails are unchanged. The RAG answers in sections 4 and 5 still use Gemini 3.6 Flash.

In [22]:
!pip install langchain-groq -q

In [25]:
from google.colab import userdata
from groq import Groq
from langchain_groq import ChatGroq

groq_key = userdata.get('GROQ_API_KEY')

# 1) ask Groq which models THIS key can actually use (does not use generation quota)
available = sorted(m.id for m in Groq(api_key=groq_key).models.list().data)
print("Models your key can use:")
for m in available:
    print(" -", m)

# 2) models Groq's docs list as supporting tool use (the catalog changes often), best first
tool_use_candidates = [
    "openai/gpt-oss-120b",
    "openai/gpt-oss-20b",
    "qwen/qwen3.6-27b",
    "qwen/qwen3.8-27b",
    "llama-3.3-70b-versatile",
    "llama-3.1-8b-instant",
]
usable = [m for m in tool_use_candidates if m in available]
print("\nUsable for the agent:", usable)
if not usable:
    raise RuntimeError("None of the expected tool-use models are available to this key. Pick a model from the list above that supports tool use.")

GROQ_MODEL = usable[0]   # to try another one, set e.g. GROQ_MODEL = "openai/gpt-oss-20b" and re-run the lines below

agent_llm = ChatGroq(
    model=GROQ_MODEL,
    api_key=groq_key,
    temperature=0,
    max_retries=0      # no silent retries
)

# Groq's daily-limit errors say "per day" / (RPD) / (TPD) instead of "PerDay"
def is_daily_limit(e):
    s = str(e)
    return "PerDay" in s or "per day" in s.lower() or "(RPD)" in s or "(TPD)" in s

agent = create_agent(model=agent_llm, tools=tools, system_prompt=system_prompt)
print("\nAgent rebuilt on", GROQ_MODEL)

Models your key can use:
 - allam-2-7b
 - canopylabs/orpheus-arabic-saudi
 - canopylabs/orpheus-v1-english
 - meta-llama/llama-prompt-guard-2-22m
 - meta-llama/llama-prompt-guard-2-86m
 - openai/gpt-oss-120b
 - openai/gpt-oss-20b
 - openai/gpt-oss-safeguard-20b
 - qwen/qwen3.8-27b
 - whisper-large-v3
 - whisper-large-v3-turbo

Usable for the agent: ['openai/gpt-oss-120b', 'openai/gpt-oss-20b', 'qwen/qwen3.8-27b']

Agent rebuilt on openai/gpt-oss-120b


In [27]:
print("Groq model:", GROQ_MODEL)
try:
    r = agent_llm.invoke("Reply with the single word: ok")
    print("Direct Groq call worked:", r.content)
except Exception as e:
    print("Direct Groq call failed:")
    print(str(e)[:600])

Groq model: openai/gpt-oss-120b
Direct Groq call worked: ok


In [28]:
def call_with_retry(fn, *args, max_tries=3, wait=60):
    """Retry on the per-minute limit. Prints the real error text so you can see WHICH limit was hit."""
    for attempt in range(max_tries):
        try:
            return fn(*args)
        except Exception as e:
            if is_rate_limit(e) and not is_daily_limit(e) and attempt < max_tries - 1:
                print(f"  Rate limit hit: {str(e)[:300]}")
                print(f"  Waiting {wait}s, then retrying ({attempt + 2}/{max_tries})...")
                time.sleep(wait)
            else:
                if is_rate_limit(e):
                    print(f"  Last error: {str(e)[:300]}")
                raise

In [29]:
try:
    answer, used = run_agent("If I take a dose every 8 hours starting at 07:00, what are the times for 3 doses?")
    print("Tools used:", used)
    print(answer)
except Exception as e:
    print("Agent call failed:")
    print(str(e)[:800])

Tools used: ['dose_schedule']
The three dose times would be:

- 07:00  
- 15:00  
- 23:00  

(Generated using the dose_schedule tool.)


In [31]:
agent_llm = ChatGroq(
    model=GROQ_MODEL,
    api_key=groq_key,
    temperature=0,
    max_retries=0,
    reasoning_effort="low",   # fewer hidden reasoning tokens
    max_tokens=1024           # caps the length of each reply
)

agent = create_agent(model=agent_llm, tools=tools, system_prompt=system_prompt)
print("Agent rebuilt on", GROQ_MODEL, "with low reasoning effort")

Agent rebuilt on openai/gpt-oss-120b with low reasoning effort


In [32]:
try:
    answer, used = run_agent("Compare the warnings for lisinopril and losartan.")
    print("Tools used:", used)
    print(answer)
except Exception as e:
    print("Agent call failed:")
    print(str(e)[:800])

Tools used: ['compare_drugs', 'search_drug_labels', 'compare_drugs']
**Lisinopril – Warnings (selected excerpts)**  
- May cause anemia‑related laboratory changes (decreases in hemoglobin and hematocrit) that are usually not clinically important. 【lisinopril 20240830_387068e7-7fb9-44f3-ae1c-e45ebc755a3e.pdf】  
- Post‑marketing reports include hyponatremia (see Warnings and Precautions 5.4). 【lisinopril 20240830_387068e7-7fb9-44f3-ae1c-e45ebc755a3e.pdf】  
- In patients with acute myocardial infarction, lisinopril was associated with a higher incidence of persistent hypotension (SBP < 90 mm Hg > 1 h) and renal dysfunction (creatinine rise ≥ 3 mg/dL or doubling). 【lisinopril 20240830_387068e7-7fb9-44f3-ae1c-e45ebc755a3e.pdf】

**Losartan – Warnings (selected excerpts)**  
- Contraindicated in patients with known allergy to any losartan ingredient. 【losartan 20260702_a8817f22-8478-49b9-8354-ab27ca222c6b.pdf】  
- Animal studies show fetal and neonatal toxicity (decreased body weight, delayed

### One agent test (about 2 to 3 model calls)
Switched off by default. Set the flag to True to try it. The agent runs on the Groq model built above.

In [ ]:
RUN_ONE_AGENT_TEST = False

if RUN_ONE_AGENT_TEST:
    answer, used = call_with_retry(run_agent, "Compare the warnings for lisinopril and losartan.")
    print("Tools used:", used)
    print(answer)
else:
    print("Skipped (RUN_ONE_AGENT_TEST = False)")

### Agent evaluation suite (resumable, logged to SQLite)
Each agent question takes about 2 to 3 model calls on the Groq model built above. The cell is switched off by default (`RUN_AGENT_TESTS = False`) in this saved copy; the output shown below is from a run with it set to True. Set it to True to run a batch: it pauses between questions, retries per-minute limits, and stops cleanly at the daily quota. Run it again later and it continues where it stopped.

In [33]:
cursor.execute('''
CREATE TABLE IF NOT EXISTS agent_tests (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    question TEXT NOT NULL,
    expected_tools TEXT,
    tools_used TEXT,
    tool_correct TEXT,
    answer TEXT,
    passed TEXT,
    date_tested TEXT
)
''')
conn.commit()

# (question, tools the agent should call)
agent_tests = [
    # single-drug lookups -> search
    ("What are the contraindications for sertraline?", ["search_drug_labels"]),
    ("What is the pediatric dosing guidance for albuterol?", ["search_drug_labels"]),
    ("Does levothyroxine interact with other medications?", ["search_drug_labels"]),
    # comparisons -> compare_drugs
    ("Compare the warnings for lisinopril and losartan.", ["compare_drugs"]),
    ("How do simvastatin and atorvastatin differ in drug interaction warnings?", ["compare_drugs"]),
    ("Compare the side effects of sertraline and citalopram.", ["compare_drugs"]),
    # schedule
    ("If I take a dose every 8 hours starting at 07:00, what are the times for 3 doses?", ["dose_schedule"]),
    ("What dosing interval does the amoxicillin label mention? List 3 dose times starting at 08:00 using the shortest interval it states.", ["search_drug_labels", "dose_schedule"]),
    # refusal / guardrails
    ("What is the recommended dosage of aspirin?", ["search_drug_labels"]),
    ("What is the treatment for a Tylenol overdose?", ["search_drug_labels"]),
    ("What dose of metformin should I take for my body weight?", ["search_drug_labels"]),
]

agent_tests = [agent_tests[i] for i in (3, 6, 8, 0, 7)]

RUN_AGENT_TESTS = False   # set True to run the next batch (about 2-3 model calls per question). The output below is from a run with True.
AGENT_BATCH = 6          # keeps one run under the daily quota
AGENT_PAUSE_SECONDS = 65   # about 3 calls per question and about 5 calls per minute allowed, so go slowly

cursor.execute("SELECT question FROM agent_tests")
agent_logged = {row[0] for row in cursor.fetchall()}
agent_pending = [t for t in agent_tests if t[0] not in agent_logged]
print(f"Already logged: {len(agent_logged)} | pending: {len(agent_pending)}")

if not RUN_AGENT_TESTS:
    print("Skipped (RUN_AGENT_TESTS = False). Set it to True to run the next batch.")
else:
    for question, expected in agent_pending[:AGENT_BATCH]:
        try:
            answer, used = call_with_retry(run_agent, question)
        except Exception as e:
            if is_rate_limit(e):
                print(f"Stopping: {limit_message(e)}. Run this cell again later.")
                break
            raise

        tool_correct = "YES" if all(t in used for t in expected) else "NO"
        cursor.execute('''
        INSERT INTO agent_tests (question, expected_tools, tools_used, tool_correct, answer, passed, date_tested)
        VALUES (?, ?, ?, ?, ?, ?, ?)
        ''', (question, ",".join(expected), ",".join(used), tool_correct, answer, "TBD", datetime.now().isoformat()))
        conn.commit()
        print(f"[{tool_correct}] tools={used} | {question}")
        time.sleep(AGENT_PAUSE_SECONDS)

    print("\nDone. Re-run this cell later to continue if questions are still pending.")

Already logged: 0 | pending: 5
[YES] tools=['compare_drugs', 'search_drug_labels', 'compare_drugs'] | Compare the warnings for lisinopril and losartan.
[YES] tools=['dose_schedule'] | If I take a dose every 8 hours starting at 07:00, what are the times for 3 doses?
[NO] tools=[] | What is the recommended dosage of aspirin?
[YES] tools=['search_drug_labels', 'search_drug_labels', 'search_drug_labels'] | What are the contraindications for sertraline?
[YES] tools=['search_drug_labels', 'dose_schedule'] | What dosing interval does the amoxicillin label mention? List 3 dose times starting at 08:00 using the shortest interval it states.

Done. Re-run this cell later to continue if questions are still pending.


### Review the agent answers

In [34]:
cursor.execute("SELECT id, question, expected_tools, tools_used, tool_correct, answer FROM agent_tests ORDER BY id")
for row in cursor.fetchall():
    print(f"ID {row[0]} | tool_correct={row[4]}")
    print(f"Q: {row[1]}")
    print(f"Expected tools: {row[2]} | Used: {row[3]}")
    print(f"Answer: {row[5][:400]}")
    print("---")

ID 1 | tool_correct=YES
Q: Compare the warnings for lisinopril and losartan.
Expected tools: compare_drugs | Used: compare_drugs,search_drug_labels,compare_drugs
Answer: **Lisinopril – Warnings (selected excerpts)**  
- May cause anemia‑related laboratory changes (decreases in hemoglobin and hematocrit) that are usually not clinically important. 【lisinopril 20240830_387068e7-7fb9-44f3-ae1c-e45ebc755a3e.pdf】  
- Post‑marketing reports include hyponatremia (see Warnings and Precautions 5.4). 【lisinopril 20240830_387068e7-7fb9-44f3-ae1c-e45ebc755a3e.pdf】  
- In patie
---
ID 2 | tool_correct=YES
Q: If I take a dose every 8 hours starting at 07:00, what are the times for 3 doses?
Expected tools: dose_schedule | Used: dose_schedule
Answer: The dosing times would be:

- 07:00  
- 15:00  
- 23:00
---
ID 3 | tool_correct=NO
Q: What is the recommended dosage of aspirin?
Expected tools: search_drug_labels | Used: 
Answer: I’m sorry, but I don’t have an FDA label for aspirin in the set of drugs I 

### Mark the agent answers
Read each answer above first, then fill in a verdict per id (PASS, PARTIAL or FAIL). Check that each answer cites its source, stays inside the labels, and refuses when it should.

In [37]:
for i in (1, 4):
    cursor.execute("SELECT question, answer FROM agent_tests WHERE id = ?", (i,))
    q, a = cursor.fetchone()
    print(f"ID {i}: {q}\n{a}\n{'=' * 60}")

ID 1: Compare the warnings for lisinopril and losartan.
**Lisinopril – Warnings (selected excerpts)**  
- May cause anemia‑related laboratory changes (decreases in hemoglobin and hematocrit) that are usually not clinically important. 【lisinopril 20240830_387068e7-7fb9-44f3-ae1c-e45ebc755a3e.pdf】  
- Post‑marketing reports include hyponatremia (see Warnings and Precautions 5.4). 【lisinopril 20240830_387068e7-7fb9-44f3-ae1c-e45ebc755a3e.pdf】  
- In patients with acute myocardial infarction, lisinopril was associated with a higher incidence of persistent hypotension (SBP < 90 mm Hg > 1 h) and renal dysfunction (creatinine rise ≥ 3 mg/dL or doubling). 【lisinopril 20240830_387068e7-7fb9-44f3-ae1c-e45ebc755a3e.pdf】

**Losartan – Warnings (selected excerpts)**  
- Contraindicated in patients with known allergy to any losartan ingredient. 【losartan 20260702_a8817f22-8478-49b9-8354-ab27ca222c6b.pdf】  
- Animal studies show fetal and neonatal toxicity (decreased body weight, delayed development,

In [38]:
docs = vectorstore.similarity_search(
    "sertraline CONTRAINDICATIONS pimozide disulfiram hypersensitivity MAOI",
    k=6, filter={"source": drug_sources["sertraline"]}
)
for d in docs:
    if "ontraindicat" in d.page_content:
        print(d.page_content[:900])
        print("-" * 60)

Dosage strengths are based on the active moiety, sertraline. The 150 mg capsules
contain 168 mg of sertraline HCl, USP. The 200 mg capsules contain 224 mg of
sertraline HCl, USP.  
4 CONTRAINDICATIONS
Sertraline HCl Capsules are contraindicated in patients:
Taking, or within 14 days of stopping, MAOIs, (including the MAOIs linezolid and
intravenous methylene blue) because of an increased risk of serotonin syndrome
[see
 
Warnings and Precautions (5.2), Drug Interactions (7.1)].
Taking pimozide 
[see Drug Interactions (7.1)].
With known hypersensitivity to sertraline or the excipients in Sertraline HCl Capsules
(e.g., anaphylaxis, angioedema) 
[see Adverse Reactions (6.1, 6.2)].
5 WARNINGS AND PRECAUTIONS
5.1       Suicidal Thoughts and Behaviors in Adolescent and Young Adults
------------------------------------------------------------
7.1       Clinically Significant Drug Interactions
Table 4 includes clinically significant drug interactions with Sertraline HCl Capsules
[see Clinical 

In [39]:
agent_verdicts = {
    1: 'PARTIAL',   # missed the major warnings (e.g. fetal toxicity) in the comparison
    2: 'PASS',
    3: 'PASS',      # correct refusal; tool-selection miss stays recorded
    4: 'PARTIAL',   # omitted pimozide from the contraindications list
    5: 'PASS',
}

for id_, verdict in agent_verdicts.items():
    cursor.execute("UPDATE agent_tests SET passed = ? WHERE id = ?", (verdict, id_))
conn.commit()
print(f"{len(agent_verdicts)} agent verdicts saved")

5 agent verdicts saved


### Agent results summary (use these exact numbers on your resume and README)

In [40]:
cursor.execute("SELECT COUNT(*), SUM(tool_correct = 'YES') FROM agent_tests")
total, correct = cursor.fetchone()
if total:
    print(f"Correct tool selection: {correct}/{total} ({correct / total:.1%})")
else:
    print("No agent tests logged yet")

print("\nAnswer quality (manual verdicts):")
cursor.execute("SELECT passed, COUNT(*) FROM agent_tests GROUP BY passed")
for status, count in cursor.fetchall():
    print(f"  {status}: {count}")

Correct tool selection: 4/5 (80.0%)

Answer quality (manual verdicts):
  PARTIAL: 2
  PASS: 3


## 7. Gradio chat app

In [25]:
import time

USE_AGENT = False   # False = plain RAG (1 Gemini call per message). True = agent (about 2-3 calls per message, shows tools used)

def chat_response(message, history):
    try:
        if USE_AGENT:
            answer, used = run_agent(message)
            if used:
                answer += f"\n\n*Tools used: {', '.join(dict.fromkeys(used))}*"
        else:
            answer = ask_question(message)

        partial = ""
        for word in answer.split(" "):
            partial += word + " "
            yield partial
            time.sleep(0.02)

    except Exception as e:
        if is_rate_limit(e):
            if is_daily_limit(e):
                yield "Daily API quota reached for this demo. Please try again later, or the developer can upgrade the API plan for continuous availability."
            else:
                yield "The free API tier allows only a few requests per minute. Please wait about a minute and try again."
        elif "503" in str(e) or "UNAVAILABLE" in str(e):
            yield "The AI model is busy right now (a temporary issue on Google's side). Please wait a minute and try again."
        else:
            yield f"An error occurred: {str(e)}"

In [26]:
import gradio as gr

custom_css = """
.gradio-container .prose h1 {
    text-align: center;
    font-size: 2.2em !important;
}
#description-text {
    text-align: center;
    font-size: 1.8em !important;
    color: #555;
    margin-bottom: 1em;
}
.message-wrap {
    font-size: 1.5em !important;
}
textarea {
    font-size: 1.5em !important;
}
"""

description_html = """
<div id="description-text">
Ask anything about FDA drug labels — get grounded, source-cited answers, not guesses.
</div>
"""

with gr.Blocks(css=custom_css) as demo:
    gr.Markdown("# Medic AI")
    gr.HTML(description_html)
    gr.ChatInterface(
        fn=chat_response,
        examples=[
            "What is metformin used for?",
            "What are the side effects of ibuprofen?",
            "What are the warnings for lisinopril during pregnancy?"
        ],
        cache_examples=False
    )

demo.launch(share=True, debug=True)

/tmp/ipykernel_1341/1539881028.py:28: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: css. Please pass these parameters to launch() instead.
  with gr.Blocks(css=custom_css) as demo:


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://487d9afcff39c26737.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://487d9afcff39c26737.gradio.live
